In [ ]:
#first of all what is zeolite
#zeolite is a porous crystal made of Si and oxygenatoms
#In this framework we will be adding Al to create a charge imbalance with in the zeolite structure
#This charge inbalance will be compensated by adding the following metals individually: Fe, Ni and Cr
#I will study the 3 metals and see how strongly each metal attracts co2 based on their enucleur charge
#One thing to keep notice of is that the reason CO2 is attracted is because of their polar C=O bonds, N2 and CH4 exc are neutral so pass through
#Pore used is 7.35A where as a CO2 molecule is 3.3A
#obtain data for the frame work from iza structure database

#Zeolite is a porous Si and O framework with pores about the size of small gas molecules.
#Swap some Si for Al, each Al leaves one minus charge. Si itself is neutral and needs nothing.
#Add positive cations to cancel those minuses, total plus must equal total minus.
#Fe²⁺ is plus 2, so it cancels two Al at once, giving 2 Al per 1 Fe²⁺.
#No two Al can sit next to each other (Löwenstein), so at most half the sites can be Al.
#CO2 sticks because its slightly negative oxygen ends are pulled to the positive cation, N2 and CH4 are too neutral so pass through.
#A higher charge, denser cation grabs CO2 harder, which is why doping with Fe beats plain Na.

In [ ]:
from pymatgen.core import Structure
from collections import deque


fau = Structure.from_file("FAU.cif")
print("Formula:", fau.composition.reduced_formula)

print("Number of atoms:", len(fau))

print("a =", round(fau.lattice.a, 3), "Å")

print("Volume =", round(fau.lattice.volume, 1), "Å³")

neighbors = fau.get_all_neighbors(3.5)
for i, site in enumerate(fau):

 if site.species_string == "Si":

  si_neigh = [n.index for n in neighbors[i] if fau[n.index].species_string == "Si"]

  print(i, "has", len(si_neigh), "Si neighbours")

  if i > 10:

   break
#this tells you the neighbouring atoms of Si

# load the framework (skip if fau is already loaded in your notebook)
fau = Structure.from_file("FAU.cif")

# build the Si-to-Si neighbour map (two Si sharing a bridging O are ~3.1 A apart)
neighbors = fau.get_all_neighbors(3.5)
adj = {}
for i, site in enumerate(fau):
    if site.species_string == "Si":
        adj[i] = [n.index for n in neighbors[i] if fau[n.index].species_string == "Si"]

# two-colour the network (chessboard): neighbours always get the opposite colour
color = {}
for start in adj:
    if start in color:
        continue
    color[start] = 0
    queue = deque([start])
    while queue:
        node = queue.popleft()
        for nb in adj[node]:
            if nb not in color:
                color[nb] = 1 - color[node]
                queue.append(nb)
            elif color[nb] == color[node]:
                print("CLASH between", node, "and", nb)

# how many sites of each colour
n0 = sum(1 for c in color.values() if c == 0)
n1 = sum(1 for c in color.values() if c == 1)
print("colour 0:", n0, "   colour 1:", n1)

# --- reload + recolour so this cell runs on its own ---
fau = Structure.from_file("FAU.cif")

neighbors = fau.get_all_neighbors(3.5)
adj = {}
for i, site in enumerate(fau):
    if site.species_string == "Si":
        adj[i] = [n.index for n in neighbors[i] if fau[n.index].species_string == "Si"]

color = {}
for start in adj:
    if start in color:
        continue
    color[start] = 0
    queue = deque([start])
    while queue:
        node = queue.popleft()
        for nb in adj[node]:
            if nb not in color:
                color[nb] = 1 - color[node]
                queue.append(nb)

# --- step 3: turn every colour-1 site into Al ---
doped = fau.copy()
for i, c in color.items():
    if c == 1:
        doped.replace(i, "Al")

# check the new composition
print("Si:", int(doped.composition["Si"]),
      " Al:", int(doped.composition["Al"]),
      " O:", int(doped.composition["O"]))
print("Si:Al ratio =", doped.composition["Si"] / doped.composition["Al"])


# --- rebuild the doped framework so this cell runs on its own ---
fau = Structure.from_file("FAU.cif")

neighbors = fau.get_all_neighbors(3.5)
adj = {}
for i, site in enumerate(fau):
    if site.species_string == "Si":
        adj[i] = [n.index for n in neighbors[i] if fau[n.index].species_string == "Si"]

color = {}
for start in adj:
    if start in color:
        continue
    color[start] = 0
    queue = deque([start])
    while queue:
        node = queue.popleft()
        for nb in adj[node]:
            if nb not in color:
                color[nb] = 1 - color[node]
                queue.append(nb)

doped = fau.copy()
for i, c in color.items():
    if c == 1:
        doped.replace(i, "Al")

# --- step 4: Lowenstein check, count Al sitting next to another Al ---
dop_neighbors = doped.get_all_neighbors(3.5)
violations = 0
for i, site in enumerate(doped):
    if site.species_string == "Al":
        al_neigh = [n.index for n in dop_neighbors[i] if doped[n.index].species_string == "Al"]
        violations += len(al_neigh)

print("Al-Al adjacencies found:", violations, "(should be 0)")

#Fe2+ isnt fixed and hence the Fe2+ can have my different sites at which it can be at aka pocket site
#i can find exact locations from other data or i can trial error using gibbs free energy to see where it is most stable

#i will get actualy data collected as previous notebooks i tried via gibbs free energy and that caused inaccuracy

nax = Structure.from_file("NaX.cif")

print(nax.composition)

#decimal amount of Al, Si and Fe so will have to adjust framework as raspa only allows whole numbers
#I a using Na because it is more accecible online rather than Fe, Na only tells us the position within the zeolite of which our Fe will be at

Formula: SiO2
Number of atoms: 576
a = 24.345 Å
Volume = 14428.8 Å³
0 has 4 Si neighbours
1 has 4 Si neighbours
2 has 4 Si neighbours
3 has 4 Si neighbours
4 has 4 Si neighbours
5 has 4 Si neighbours
6 has 4 Si neighbours
7 has 4 Si neighbours
8 has 4 Si neighbours
9 has 4 Si neighbours
10 has 4 Si neighbours
11 has 4 Si neighbours
colour 0: 96    colour 1: 96
Si: 96  Al: 96  O: 384
Si:Al ratio = 1.0
Al-Al adjacencies found: 0 (should be 0)
Na91.36 Si103.68 Al88.32 O384


c:\Users\alans\anaconda3\envs\calphad\Lib\site-packages\pymatgen\io\cif.py:1053: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  self.symmetry_operations = self.get_symops(data)  # type:ignore[assignment]
c:\Users\alans\anaconda3\envs\calphad\Lib\site-packages\pymatgen\core\structure.py:3172: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
Skipping relative stoichiometry check because CIF does not contain formula keys.
  struct = parser.parse_structures(primitive=primitive)[0]
